In [ ]:
# -*- coding: utf-8 -*-

USE_GOOGLE_DRIVE = True

ZIP_PATH = "/content/drive/MyDrive/ML_Project/project_files/First_benchmark/tiny-imagenet-processed.zip"

DATA_ROOT = "/content/drive/MyDrive/ML_Project/data/TINYIMG"

SAVE_DIR = "/content/drive/MyDrive/ML_Project/project_files/Group_norm"

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

import os, random, zipfile
from typing import Dict, List
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Subset, DataLoader, Dataset
from torchvision import transforms
from copy import deepcopy

SEED = 42
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
NUM_WORKERS = 0

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = True


def get_best_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_best_device()

PIN_MEM = (device.type == "cuda")

print(f"[INFO] Using device: {device}")

def ensure_extracted(zip_path: str, data_root: str) -> str:
    processed_dir = os.path.join(data_root, "processed")
    if os.path.isdir(processed_dir) and len(os.listdir(processed_dir)) > 0:
        print(f"[INFO] Found processed data at: {processed_dir}")
        return data_root

    if not os.path.isfile(zip_path):
        raise FileNotFoundError(
            f"ZIP not found at:\n{zip_path}\nPlease update ZIP_PATH to the correct path of tiny-imagenet-processed.zip."
        )

    print(f"[INFO] Extracting ZIP from:\n{zip_path}\n-> to:\n{data_root}")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(data_root)
    processed_dir = os.path.join(data_root, "processed")
    assert os.path.isdir(processed_dir), "processed/ folder missing after unzip!"
    print(f"[INFO] Extracted. processed/ ready at: {processed_dir}")
    return data_root

DATA_ROOT = ensure_extracted(ZIP_PATH, DATA_ROOT)

# =========[ 4) Dataset (processed .npy) + Augmentations ]=========
TIN_IMAGENET_MEAN = (0.4802, 0.4480, 0.3975)
TIN_IMAGENET_STD  = (0.2770, 0.2691, 0.2821)

class TinyImagenet(Dataset):

    def __init__(self, root: str, train: bool=True, transform: transforms = None):
        self.root = root
        self.train = train
        self.transform = transform

        split = "train" if self.train else "val"
        xs, ys = [], []
        for num in range(20):
            xs.append(np.load(os.path.join(root, f'processed/x_{split}_{num+1:02d}.npy')))
            ys.append(np.load(os.path.join(root, f'processed/y_{split}_{num+1:02d}.npy')))
        self.data = np.concatenate(np.array(xs))
        self.targets = np.concatenate(np.array(ys)).astype(int)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        img, target = self.data[index], int(self.targets[index])
        img = Image.fromarray(np.uint8(255 * img))
        if self.transform is not None:
            img = self.transform(img)
        return img, target

def get_tiny_transforms():
    tf_train = transforms.Compose([
        transforms.RandomCrop(64, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(TIN_IMAGENET_MEAN, TIN_IMAGENET_STD),
    ])
    tf_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(TIN_IMAGENET_MEAN, TIN_IMAGENET_STD),
    ])
    return tf_train, tf_test

def get_tiny_datasets():
    tf_train, tf_test = get_tiny_transforms()
    train_set = TinyImagenet(root=DATA_ROOT, train=True,  transform=tf_train)
    test_set  = TinyImagenet(root=DATA_ROOT, train=False, transform=tf_test)
    print(f"[INFO] Train size: {len(train_set)} | Val size: {len(test_set)}")
    return train_set, test_set

# =========[ 5) ResNet-18 (GroupNorm) + Multi-Head ]=========

def conv3x3(in_planes: int, out_planes: int, stride: int = 1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=1, bias=False)

def _gn(num_channels: int, num_groups: int = 32):
    return nn.GroupNorm(num_groups, num_channels)

class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes: int, planes: int, stride: int = 1):
        super().__init__()
        self.conv1 = conv3x3(in_planes, planes, stride)
        self.gn1   = _gn(planes)
        self.conv2 = conv3x3(planes, planes, 1)
        self.gn2   = _gn(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes * self.expansion, kernel_size=1,
                          stride=stride, bias=False),
                _gn(planes * self.expansion)
            )

    def forward(self, x):
        out = torch.relu(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        out = out + self.shortcut(x)
        out = torch.relu(out)
        return out

class ResNet18Backbone(nn.Module):
    def __init__(self, nf: int = 64):
        super().__init__()
        self.nf = nf
        self.in_planes = nf
        self.expansion = 1
        self.conv1 = conv3x3(3, nf)
        self.gn1   = _gn(nf)
        self.layer1 = self._make_layer(BasicBlock, nf,     2, stride=1)
        self.layer2 = self._make_layer(BasicBlock, nf * 2, 2, stride=2)
        self.layer3 = self._make_layer(BasicBlock, nf * 4, 2, stride=2)
        self.layer4 = self._make_layer(BasicBlock, nf * 8, 2, stride=2)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers, in_planes = [], self.in_planes
        for s in strides:
            layers.append(block(in_planes, planes, s))
            in_planes = planes * block.expansion
        self.in_planes = in_planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = torch.relu(self.gn1(self.conv1(x)))
        out = self.layer1(out); out = self.layer2(out)
        out = self.layer3(out); out = self.layer4(out)
        out = torch.nn.functional.avg_pool2d(out, out.shape[2])
        feat = out.view(out.size(0), -1)
        return feat

    @property
    def out_dim(self) -> int:
        return self.nf * 8 * self.expansion

class MultiHeadNet(nn.Module):
    def __init__(self, backbone: ResNet18Backbone):
        super().__init__()
        self.backbone = backbone
        self.heads = nn.ModuleDict()

    def add_head(self, task_name: str, num_classes: int):
        if task_name in self.heads:
            raise ValueError(f"Head '{task_name}' already exists.")
        head = nn.Linear(self.backbone.out_dim, num_classes)
        head = head.to(next(self.backbone.parameters()).device)
        self.heads[task_name] = head

    def forward(self, x, task_name: str):
        if task_name not in self.heads:
            raise ValueError(f"Head '{task_name}' not found")
        feat = self.backbone(x)
        return self.heads[task_name](feat)

def build_tasks(num_classes: int = 200, classes_per_task: int = 20) -> Dict[str, List[int]]:
    assert num_classes % classes_per_task == 0
    n_tasks = num_classes // classes_per_task  # 10
    tasks = {}
    for t in range(n_tasks):
        start = t * classes_per_task
        tasks[f"task{t+1}"] = list(range(start, start + classes_per_task))
    return tasks

def filter_indices_by_classes(dataset, classes: List[int]) -> List[int]:
    t = dataset.targets
    return [i for i, y in enumerate(t) if int(y) in classes]

def remap_labels(original_targets: List[int], keep_classes: List[int]) -> List[int]:
    class_to_new = {c: i for i, c in enumerate(sorted(keep_classes))}
    return [class_to_new[int(y)] for y in original_targets]

class RelabeledSubset(Subset):
    def __init__(self, dataset, indices: List[int], keep_classes: List[int]):
        super().__init__(dataset, indices)

        original_targets = [int(dataset.targets[i]) for i in indices]
        self.new_targets = remap_labels(original_targets, keep_classes)

    def __getitem__(self, idx):
        x, _ = super().__getitem__(idx)
        y_new = self.new_targets[idx]
        return x, y_new

    def __getitems__(self, indices):
        return [self.__getitem__(idx) for idx in indices]

def make_task_subset(dataset, task_classes: List[int]) -> RelabeledSubset:
    idx = filter_indices_by_classes(dataset, task_classes)
    return RelabeledSubset(dataset, idx, task_classes)

def make_loader(ds, batch_size: int, shuffle: bool) -> DataLoader:
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEM)

# =========[ 7) Evaluation ]=========
@torch.no_grad()
def evaluate(model: MultiHeadNet, task_name: str, loader: DataLoader) -> float:
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        preds = model(x, task_name).argmax(1)
        correct += (preds == y).sum().item()
        total += x.size(0)
    return correct / max(1, total)

# =========[ 8) Train One Configuration ]=========
def train_one_config(model: MultiHeadNet,
                     task_name: str,
                     train_loader: DataLoader,
                     val_loader: DataLoader,   # test_loader
                     epochs: int,
                     lr: float):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)

    best_val = 0.0
    best_state = None
    best_epoch = -1

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_samples = 0

        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad()
            logits = model(x, task_name)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * x.size(0)
            total_correct += (logits.argmax(1) == y).sum().item()
            total_samples += x.size(0)

        avg_loss = total_loss / max(1, total_samples)
        train_acc = total_correct / max(1, total_samples)


        val_acc = evaluate(model, task_name, val_loader)
        if val_acc > best_val:
            best_val = val_acc
            best_epoch = epoch
            best_state = deepcopy(model.state_dict())

        print(f"Epoch {epoch:03d} | Train Loss: {avg_loss:.4f} | "
              f"Train Acc: {train_acc*100:.2f}% | Test Acc: {val_acc*100:.2f}% "
              f"(Best: {best_val*100:.2f}% @epoch {best_epoch})")

    return best_val, best_state, best_epoch

# =========[ 9) Grid-Search ]=========
def grid_search_task1():
    tasks = build_tasks(num_classes=200, classes_per_task=20)
    task1_classes = tasks["task1"]  # [0..19]

    # datasets
    train_set, test_set = get_tiny_datasets()

    task1_train = make_task_subset(train_set, task1_classes)
    task1_test  = make_task_subset(test_set,  task1_classes)
    print("Total TRAIN (task1):", len(task1_train))
    print("Total TEST  (task1):", len(task1_test))

    # model + heads
    backbone = ResNet18Backbone(nf=64)
    model = MultiHeadNet(backbone=backbone)
    for i in range(1, 11):
        model.add_head(f"task{i}", num_classes=20)
    model.to(device)

    # search space
    batch_sizes = [32]
    epochs_list = [350]
    lrs = [0.01]

    best_cfg = None
    best_val_global = -1.0
    best_state_global = None
    best_epoch_global = -1
    init_state = deepcopy(model.state_dict())

    for bs in batch_sizes:
        train_loader = make_loader(task1_train, batch_size=bs, shuffle=True)
        test_loader  = make_loader(task1_test,  batch_size=bs, shuffle=False)
        for epochs in epochs_list:
            for lr in lrs:
                print(f"\n=== Trying: bs={bs}, epochs={epochs}, lr={lr} ===")
                model.load_state_dict(deepcopy(init_state)); model.to(device)

                best_val_this, best_state_this, best_epoch_this = \
                    train_one_config(model, "task1", train_loader, test_loader, epochs, lr)

                if best_val_this > best_val_global:
                    best_val_global = best_val_this
                    best_cfg = {"batch_size": bs, "epochs": epochs, "lr": lr}
                    best_state_global = deepcopy(best_state_this)
                    best_epoch_global = best_epoch_this

    print("\nGrid-Search DONE.")
    print(f"Best Test: {best_val_global*100:.2f}% with {best_cfg} @epoch {best_epoch_global}")

    if best_state_global is None:
        best_state_global = model.state_dict()

    ckpt_path = os.path.join(SAVE_DIR, "expriment2_GN_task1_best_test_for_finetune_tiny_imageNet_temp.pth")
    torch.save(best_state_global, ckpt_path)
    print(f"Saved weights only -> {ckpt_path}")

# =========[ 10) Main ]=========
if __name__ == "__main__":
    grid_search_task1()


Mounted at /content/drive
[INFO] Using device: cuda
[INFO] Found processed data at: /content/drive/MyDrive/ML_Project/data/TINYIMG/processed
[INFO] Train size: 100000 | Val size: 10000
Total TRAIN (task1): 10000
Total TEST  (task1): 1000

=== Trying: bs=32, epochs=350, lr=0.01 ===
Epoch 001 | Train Loss: 3.0652 | Train Acc: 6.64% | Test Acc: 10.60% (Best: 10.60% @epoch 1)
Epoch 002 | Train Loss: 2.7495 | Train Acc: 13.87% | Test Acc: 16.20% (Best: 16.20% @epoch 2)
Epoch 003 | Train Loss: 2.5790 | Train Acc: 19.59% | Test Acc: 21.80% (Best: 21.80% @epoch 3)
Epoch 004 | Train Loss: 2.4170 | Train Acc: 24.73% | Test Acc: 26.10% (Best: 26.10% @epoch 4)
Epoch 005 | Train Loss: 2.2827 | Train Acc: 28.26% | Test Acc: 27.10% (Best: 27.10% @epoch 5)
Epoch 006 | Train Loss: 2.1846 | Train Acc: 30.85% | Test Acc: 27.30% (Best: 27.30% @epoch 6)
Epoch 007 | Train Loss: 2.0581 | Train Acc: 34.94% | Test Acc: 37.60% (Best: 37.60% @epoch 7)
Epoch 008 | Train Loss: 1.9397 | Train Acc: 38.86% | Test Acc